# Cross-Industry Accelerator — Pipeline Orchestrator
### One-Click End-to-End Deployment

This notebook **orchestrates the entire 8-notebook pipeline** with centralized logging,
error handling, and a full audit trail.

| What it does | Detail |
|---|---|
| **Runs notebooks 00–07** in sequence | Each via `notebookutils.notebook.run()` |
| **Tracks every record** | Row counts per table, per target (Lakehouse/Warehouse/Eventhouse) |
| **Logs every artifact** | Semantic Model, Ontology, Agents, Dashboards — with IDs |
| **Captures errors** | Structured error log with stack traces and fatal/non-fatal classification |
| **Generates audit report** | Full pipeline execution summary at the end |
| **Persists to Lakehouse** | Writes `pipeline_runs`, `pipeline_events`, `pipeline_artifacts`, `pipeline_errors` Delta tables |

> **Usage:** Set `INDUSTRY` below, then **Run All**. That's it.
>
> **Prerequisites:** Same as the individual notebooks — Fabric workspace with Lakehouse, Warehouse, Eventhouse,
> CSV data uploaded, and ontology package + WHL in Lakehouse Files.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PIPELINE CONFIGURATION — Set your industry and run options
# ════════════════════════════════════════════════════════════════════════

# ── REQUIRED: Set your industry key ────────────────────────────────────
INDUSTRY = "advertising"  # Change this to your target industry

# Supported: "healthcare", "finance", "insurance", "retail", "cpg",
#            "construction", "oil_and_gas", "media", "law_firms",
#            "telecom", "advertising"

# ── OPTIONAL: Pipeline run options ─────────────────────────────────────
SKIP_ON_FAILURE = False       # True = skip failed steps, continue pipeline
                               # False = stop pipeline on first fatal error

PERSIST_LOG = True             # Write audit log to Lakehouse Delta tables

NOTEBOOK_TIMEOUT = 1800        # Max seconds per notebook (30 min default)

# ── OPTIONAL: Skip specific steps (for re-runs) ───────────────────────
# Set to True to skip a step (e.g., skip ingestion if data already loaded)
SKIP_STEPS = {
    "00_Industry_Config":       False,
    "01_Data_Ingestion":        False,
    "02_Load_Lakehouse":        False,
    "03_Load_Warehouse":        False,
    "04_Create_Semantic_Model":  False,
    "05_Create_Ontology":       False,
    "06_Create_Data_Agent":     False,
    "07_Create_Dashboards":     False,
}

print(f"Pipeline configured for: {INDUSTRY}")
print(f"  Skip on failure: {SKIP_ON_FAILURE}")
print(f"  Persist log:     {PERSIST_LOG}")
print(f"  Timeout/step:    {NOTEBOOK_TIMEOUT}s")
skipped = [k for k, v in SKIP_STEPS.items() if v]
if skipped:
    print(f"  Skipping:        {', '.join(skipped)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD PIPELINE LOGGER
# ════════════════════════════════════════════════════════════════════════

%run Pipeline_Logger

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PREFLIGHT CHECKS — Validate environment before running pipeline
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric

log_pipeline_event("PIPELINE_INIT", INDUSTRY, f"run_id={PIPELINE_RUN_ID}")

# Validate industry
SUPPORTED = {
    "healthcare", "finance", "insurance", "retail", "cpg",
    "construction", "oil_and_gas", "media", "law_firms",
    "telecom", "advertising"
}
if INDUSTRY not in SUPPORTED:
    raise ValueError(f"Invalid INDUSTRY='{INDUSTRY}'. Supported: {sorted(SUPPORTED)}")

# Validate Fabric runtime
try:
    workspace_id = fabric.get_workspace_id()
    log_pipeline_event("PREFLIGHT", "workspace", f"id={workspace_id}")
    print(f"  ✓ Workspace: {workspace_id}")
except Exception as e:
    raise RuntimeError(f"Not running in Fabric environment: {e}")

# Validate notebooks exist in workspace
items_df = fabric.list_items()
notebooks_in_ws = set(items_df[items_df["Type"] == "Notebook"]["Display Name"].tolist())

required_notebooks = [
    "00_Industry_Config", "01_Data_Ingestion", "02_Load_Lakehouse",
    "03_Load_Warehouse", "04_Create_Semantic_Model", "05_Create_Ontology",
    "06_Create_Data_Agent", "07_Create_Dashboards",
    "ZT_Security_Utils", "Pipeline_Logger",
]

missing = [nb for nb in required_notebooks if nb not in notebooks_in_ws]
if missing:
    print(f"  ⚠ Missing notebooks: {missing}")
    print(f"    Import all notebooks from the repo into your Fabric workspace.")
    raise RuntimeError(f"Missing required notebooks: {missing}")

log_pipeline_event("PREFLIGHT", "notebooks", f"All {len(required_notebooks)} required notebooks found")
print(f"  ✓ All {len(required_notebooks)} required notebooks present")
print(f"\n✅ Preflight checks passed — ready to run pipeline")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PIPELINE EXECUTION ENGINE
# ════════════════════════════════════════════════════════════════════════

import time

def run_pipeline_step(step_info):
    """
    Execute a single pipeline notebook with full logging.
    Returns True on success, False on failure.
    """
    step_name = step_info["step"]
    description = step_info["description"]

    # Check if step should be skipped
    if SKIP_STEPS.get(step_name, False):
        log_pipeline_event("STEP_SKIPPED", step_name, "Skipped by user configuration")
        print(f"\n  ⏭ SKIPPED: {step_name} (configured to skip)")
        return True

    pipeline_step_start(step_name, description)

    try:
        # Run the notebook via Fabric's notebook orchestration API
        exit_value = notebookutils.notebook.run(
            step_name,
            NOTEBOOK_TIMEOUT,
            {"INDUSTRY": INDUSTRY}  # Pass industry as parameter
        )

        # Capture exit value if returned
        if exit_value:
            log_pipeline_event("STEP_OUTPUT", step_name, str(exit_value)[:500])

        pipeline_step_complete(step_name)
        return True

    except Exception as e:
        pipeline_step_fail(step_name, e, fatal=not SKIP_ON_FAILURE)
        return False


print("Pipeline execution engine loaded.")
print(f"  Steps to run: {sum(1 for s in PIPELINE_STEPS if not SKIP_STEPS.get(s['step'], False))}")
print(f"  Steps to skip: {sum(1 for s in PIPELINE_STEPS if SKIP_STEPS.get(s['step'], False))}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN THE FULL PIPELINE
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'╔'+'═'*68+'╗'}")
print(f"{'║'} CROSS-INDUSTRY ACCELERATOR PIPELINE{' '*32}{'║'}")
print(f"{'║'} Industry: {INDUSTRY:<57}{'║'}")
print(f"{'║'} Run ID:   {PIPELINE_RUN_ID:<57}{'║'}")
print(f"{'╚'+'═'*68+'╝'}")

pipeline_failed = False

for step_info in PIPELINE_STEPS:
    step_name = step_info["step"]

    # If pipeline already failed and we're not in skip-on-failure mode, stop
    if pipeline_failed and not SKIP_ON_FAILURE:
        log_pipeline_event("STEP_SKIPPED", step_name, "Pipeline halted due to prior failure")
        print(f"\n  ⏹ HALTED: {step_name} — pipeline stopped due to prior failure")
        continue

    success = run_pipeline_step(step_info)

    if not success:
        pipeline_failed = True
        if not SKIP_ON_FAILURE:
            print(f"\n  🛑 Pipeline halted at {step_name}. Set SKIP_ON_FAILURE=True to continue past errors.")

    # Brief pause between steps to allow Fabric API state to propagate
    if not SKIP_STEPS.get(step_name, False):
        time.sleep(5)

print(f"\n{'━'*65}")
if pipeline_failed:
    print(f"⚠ Pipeline completed with errors. See summary below.")
else:
    print(f"✅ Pipeline completed successfully for {INDUSTRY}!")
print(f"{'━'*65}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# COLLECT POST-RUN TELEMETRY
# ════════════════════════════════════════════════════════════════════════
# After all notebooks have run, collect inventory of what was created.
# This gives us the definitive artifact registry and record counts.

import sempy.fabric as fabric

print("Collecting post-run telemetry...")

try:
    # Re-run config to get all variable names
    %run 00_Industry_Config

    workspace_id = fabric.get_workspace_id()
    items_df = fabric.list_items()

    # ── Count records in Lakehouse Delta tables ──
    print(f"\n  Lakehouse record counts:")
    for table_name in LAKEHOUSE_TABLES:
        try:
            row_count = spark.read.table(table_name).count()
            log_records_loaded("02_Load_Lakehouse", table_name, row_count, target="Lakehouse")
            print(f"    {table_name:<45} {row_count:>8,} rows")
        except Exception:
            pass  # Table may not exist if step was skipped

    # ── Count records in Eventhouse (if configured) ──
    if EVENTHOUSE_CLUSTER_URI and EVENTHOUSE_DATABASE:
        import requests
        access_token = notebookutils.credentials.getToken('kusto')
        print(f"\n  Eventhouse record counts:")
        for table_name in EVENTHOUSE_TABLES:
            kql_table = table_name.replace("fact_", "").replace("stream_", "")
            try:
                resp = requests.post(
                    f"{EVENTHOUSE_CLUSTER_URI}/v1/rest/query",
                    headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
                    json={"db": EVENTHOUSE_DATABASE, "csl": f"{kql_table} | count"},
                    timeout=30
                )
                if resp.status_code == 200:
                    count_val = resp.json().get("Tables", [{}])[0].get("Rows", [[0]])[0][0]
                    log_records_loaded("02_Load_Lakehouse", kql_table, count_val, target="Eventhouse")
                    print(f"    {kql_table:<45} {count_val:>8,} rows")
            except Exception:
                pass

    # ── Verify Fabric artifacts ──
    print(f"\n  Fabric artifacts in workspace:")
    artifact_checks = [
        ("Lakehouse",     LAKEHOUSE_NAME,      "00_Industry_Config"),
        ("Warehouse",     WAREHOUSE_NAME,       "03_Load_Warehouse"),
        ("SemanticModel", SEMANTIC_MODEL_NAME,   "04_Create_Semantic_Model"),
    ]
    # Add optional artifacts
    if EVENTHOUSE_NAME:
        artifact_checks.append(("Eventhouse", EVENTHOUSE_NAME, "02_Load_Lakehouse"))
    try:
        artifact_checks.append(("Ontology", ONTOLOGY_NAME, "05_Create_Ontology"))
    except NameError:
        pass

    for art_type, art_name, step in artifact_checks:
        match = items_df[items_df["Display Name"] == art_name]
        if not match.empty:
            art_id = str(match.iloc[0]["Id"])
            log_artifact_created(step, art_type, art_name, art_id)
        else:
            print(f"    ⚠ {art_type}: {art_name} — NOT FOUND")

    # Check for agents
    for agent_name, step in [(DATA_AGENT_NAME, "06_Create_Data_Agent"), (OPS_AGENT_NAME, "06_Create_Data_Agent")]:
        match = items_df[items_df["Display Name"] == agent_name]
        if not match.empty:
            log_artifact_created(step, "DataAgent", agent_name, str(match.iloc[0]["Id"]))

    # Check for dashboards/reports
    rt_name = f"{INDUSTRY_LABEL}_RealTime_Dashboard"
    rpt_name = f"{INDUSTRY_LABEL}_Analytics_Report"
    for dash_name, dash_type, step in [
        (rt_name, "KQLDashboard", "07_Create_Dashboards"),
        (rpt_name, "Report", "07_Create_Dashboards"),
    ]:
        match = items_df[items_df["Display Name"] == dash_name]
        if not match.empty:
            log_artifact_created(step, dash_type, dash_name, str(match.iloc[0]["Id"]))

    print(f"\n✅ Post-run telemetry collected.")

except Exception as e:
    print(f"  ⚠ Could not collect full telemetry: {e}")
    print(f"    Pipeline summary will use data from step-level logging.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ZT SECURITY AUDIT — Merge ZT audit events into pipeline log
# ════════════════════════════════════════════════════════════════════════

try:
    zt_log = get_audit_log()  # From ZT_Security_Utils (loaded by 00_Industry_Config)
    if zt_log:
        print(f"  Merging {len(zt_log)} Zero Trust audit events into pipeline log...")
        for entry in zt_log:
            log_pipeline_event(
                f"ZT_{entry.get('action', 'UNKNOWN')}",
                entry.get('target', ''),
                entry.get('details', ''),
                status=entry.get('status', 'OK')
            )
        print(f"  ✅ {len(zt_log)} ZT audit events merged.")
except Exception as e:
    print(f"  ⚠ Could not merge ZT audit log: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PIPELINE EXECUTION REPORT
# ════════════════════════════════════════════════════════════════════════

summary = print_pipeline_summary()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PERSIST AUDIT LOG TO LAKEHOUSE (for compliance & historical tracking)
# ════════════════════════════════════════════════════════════════════════

if PERSIST_LOG:
    print("Persisting pipeline audit log to Lakehouse Delta tables...")
    print("  Tables: pipeline_runs, pipeline_events, pipeline_artifacts, pipeline_errors")
    persist_pipeline_log(spark, lakehouse_schema=LAKEHOUSE_SCHEMA)
    print(f"\n  ✅ Audit log persisted. Query with:")
    print(f"     SELECT * FROM {LAKEHOUSE_SCHEMA}.pipeline_runs WHERE run_id = '{PIPELINE_RUN_ID}'")
    print(f"     SELECT * FROM {LAKEHOUSE_SCHEMA}.pipeline_events WHERE run_id = '{PIPELINE_RUN_ID}' ORDER BY timestamp")
else:
    print("Log persistence disabled (PERSIST_LOG=False).")
    print("In-memory log available via: get_pipeline_summary()")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DEPLOYMENT COMPLETE — Final status
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'╔'+'═'*68+'╗'}")
print(f"{'║'} DEPLOYMENT COMPLETE{' '*49}{'║'}")
print(f"{'╠'+'═'*68+'╣'}")
print(f"{'║'} Industry:   {INDUSTRY:<55}{'║'}")
print(f"{'║'} Run ID:     {PIPELINE_RUN_ID:<55}{'║'}")
print(f"{'║'} Status:     {summary['status']:<55}{'║'}")
print(f"{'║'} Duration:   {summary['elapsed']:<55}{'║'}")
print(f"{'║'} Rows:       {summary['total_rows_processed']:<55,}{'║'}")
print(f"{'║'} Artifacts:  {summary['total_artifacts_created']:<55}{'║'}")
print(f"{'║'} Errors:     {summary['total_errors']:<55}{'║'}")
print(f"{'╠'+'═'*68+'╣'}")
print(f"{'║'} Artifacts Created:{' '*50}{'║'}")

for a in _ARTIFACT_REGISTRY:
    line = f"  📦 {a['type']}: {a['name']}"
    print(f"{'║'} {line:<67}{'║'}")

print(f"{'╠'+'═'*68+'╣'}")
print(f"{'║'} Next Steps:{' '*57}{'║'}")
print(f"{'║'}   • Open the KQL Dashboard for real-time monitoring{' '*17}{'║'}")
print(f"{'║'}   • Open the Power BI Report for historical analysis{' '*16}{'║'}")
print(f"{'║'}   • Ask the QA Agent industry-specific questions{' '*20}{'║'}")
print(f"{'║'}   • Ask the Ops Agent to monitor live events{' '*24}{'║'}")
print(f"{'╚'+'═'*68+'╝'}")

# Pipeline run summary
print(f"\n  Pipeline Summary:")
print(f"  00 → Industry Config           {'✅' if _STEP_LOG.get('00_Industry_Config', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('00_Industry_Config') else '❌'}")
print(f"  01 → Data Ingestion            {'✅' if _STEP_LOG.get('01_Data_Ingestion', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('01_Data_Ingestion') else '❌'}")
print(f"  02 → Lakehouse + Eventhouse     {'✅' if _STEP_LOG.get('02_Load_Lakehouse', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('02_Load_Lakehouse') else '❌'}")
print(f"  03 → Warehouse                 {'✅' if _STEP_LOG.get('03_Load_Warehouse', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('03_Load_Warehouse') else '❌'}")
print(f"  04 → Semantic Model            {'✅' if _STEP_LOG.get('04_Create_Semantic_Model', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('04_Create_Semantic_Model') else '❌'}")
print(f"  05 → Ontology                  {'✅' if _STEP_LOG.get('05_Create_Ontology', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('05_Create_Ontology') else '❌'}")
print(f"  06 → AI Agents                 {'✅' if _STEP_LOG.get('06_Create_Data_Agent', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('06_Create_Data_Agent') else '❌'}")
print(f"  07 → Dashboards                {'✅' if _STEP_LOG.get('07_Create_Dashboards', {}).get('status') == 'SUCCESS' else '⏭' if SKIP_STEPS.get('07_Create_Dashboards') else '❌'}")
print(f"  ── → Pipeline Orchestrator     ← YOU ARE HERE")